# Notebook 00: Physics RLVR — Data Preparation

Partitions `TIGER-Lab/MMLU-Pro` into three non-overlapping splits for the SFT → GRPO → Eval pipeline.

| File | Source | Size | Purpose |
|------|--------|------|---------|
| `sft_train.jsonl` | validation split, all categories | 70 | SFT warmup — includes `cot_content` CoT traces |
| `grpo_train.jsonl` | test split, physics only | 1,000 | GRPO training — no CoT traces needed |
| `eval.jsonl` | test split, physics only | 200 | Held-out evaluation — never seen during training |

**Key design choices:**
- SFT uses the entire validation split (all 14 categories, not just physics) because warmup is format-only — the model is learning the XML output schema, not physics. Domain doesn't matter.
- GRPO and eval draw exclusively from the physics test split, with a fixed `seed=42` shuffle to ensure the 1,000/200 boundary is reproducible.
- No tokenization or chat-template formatting here — that happens in the training notebooks.

## Step 1: Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/grpo-verified-reasoner
!ls

Mounted at /content/drive
/content/drive/MyDrive/grpo-verified-reasoner
data			      models	    README.md			 wandb
grpo_trainer_lora_model       notebooks     src
huggingface_tokenizers_cache  outputs	    unsloth_compiled_cache
LICENSE			      physics_rlvr  _unsloth_sentencepiece_temp


In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Step 2: Install Dependencies

In [3]:
#!pip install -q datasets

## Step 3: Load MMLU-Pro

In [4]:
from datasets import load_dataset

raw = load_dataset("TIGER-Lab/MMLU-Pro")
print("Splits:", list(raw.keys()))
print("Columns:", raw[list(raw.keys())[0]].column_names)
for split in raw:
    print(f"  {split}: {len(raw[split])} rows")

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.14M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/42.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/12032 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/70 [00:00<?, ? examples/s]

Splits: ['test', 'validation']
Columns: ['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src']
  test: 12032 rows
  validation: 70 rows


## Step 4: Build SFT Split (Validation, All Categories)

`cot_content` is only populated in the `validation` split of MMLU-Pro.
We filter to rows with non-empty `cot_content` across **all categories** — mixed domain
is fine because SFT warmup is format-only (teaching the XML schema, not physics).

In [5]:
from collections import Counter

val = raw["validation"]
print(f"Validation split: {len(val)} total rows")
print("Categories:", Counter(val["category"]).most_common())

# Rows with populated cot_content
n_with_cot = sum(1 for x in val if isinstance(x["cot_content"], str) and len(x["cot_content"]) > 0)
print(f"\nRows with non-empty cot_content: {n_with_cot} / {len(val)}")

Validation split: 70 total rows
Categories: [('math', 5), ('health', 5), ('physics', 5), ('business', 5), ('biology', 5), ('chemistry', 5), ('computer science', 5), ('economics', 5), ('engineering', 5), ('philosophy', 5), ('other', 5), ('history', 5), ('psychology', 5), ('law', 5)]

Rows with non-empty cot_content: 70 / 70


In [ ]:
VALID_LETTERS = set("ABCDEFGHIJ")  # MMLU-Pro supports up to 10 options (A–J)

# Keep only rows with a non-empty CoT trace
sft_data = val.filter(
    lambda x: isinstance(x["cot_content"], str) and len(x["cot_content"]) > 0
)
print(f"SFT rows after cot_content filter: {len(sft_data)}")
assert len(sft_data) >= 50, f"Too few SFT rows after filter: {len(sft_data)}"

# Schema assertions — catch malformed rows before they silently corrupt training
for i, ex in enumerate(sft_data):
    assert 2 <= len(ex["options"]) <= 10, \
        f"SFT row {i}: unexpected option count {len(ex['options'])}"
    assert ex["answer"].upper() in VALID_LETTERS, \
        f"SFT row {i}: bad answer letter '{ex['answer']}'"

# Normalize answers to uppercase so reward functions always see "A" not "a"
sft_data = sft_data.map(lambda x: {"answer": x["answer"].upper()})

print(f"Schema assertions passed on all {len(sft_data)} SFT rows.")
print("Category breakdown:", Counter(sft_data["category"]).most_common())

sft_list = list(sft_data)  # Convert to plain list for downstream JSONL serialization

## Step 5: Build GRPO + Eval Splits (Test, Physics Only)

The test split contains 1,299 physics rows. We shuffle with `seed=42` and take the first 1,000 for GRPO training and the next 200 for evaluation. The remaining 99 are discarded.

The eval split is strictly held out — it overlaps with neither the SFT validation data nor the GRPO training data. This guarantees that evaluation accuracy reflects genuine generalization.

In [7]:
physics = raw["test"].filter(lambda x: x["category"] == "physics")
print(f"Physics rows in test split: {len(physics)}")
assert len(physics) >= 1200, f"Expected >= 1200 physics rows, got {len(physics)}"

# Audit option counts
option_counts = Counter(len(ex["options"]) for ex in physics)
print("Option count distribution:", dict(sorted(option_counts.items())))

# Schema assertions
for i, ex in enumerate(physics):
    assert 2 <= len(ex["options"]) <= 10, \
        f"Physics row {i}: unexpected option count {len(ex['options'])}"
    assert ex["answer"].upper() in VALID_LETTERS, \
        f"Physics row {i}: bad answer letter '{ex['answer']}'"

print(f"Schema assertions passed on all {len(physics)} physics rows.")

# Normalize answers to uppercase
physics = physics.map(lambda x: {"answer": x["answer"].upper()})

Filter:   0%|          | 0/12032 [00:00<?, ? examples/s]

Physics rows in test split: 1299
Option count distribution: {4: 6, 5: 1, 6: 4, 7: 4, 8: 14, 9: 28, 10: 1242}
Schema assertions passed on all 1299 physics rows.


Map:   0%|          | 0/1299 [00:00<?, ? examples/s]

In [8]:
# Shuffle and split
physics = physics.shuffle(seed=42)
physics_list = list(physics)

grpo_split = physics_list[:1000]
eval_split = physics_list[1000:1200]

assert len(grpo_split) == 1000, f"Expected 1000 GRPO examples, got {len(grpo_split)}"
assert len(eval_split) == 200,  f"Expected 200 eval examples, got {len(eval_split)}"

print(f"SFT split:  {len(sft_list)} examples  (validation, all categories)")
print(f"GRPO split: {len(grpo_split)} examples (test, physics)")
print(f"Eval split: {len(eval_split)} examples  (test, physics, held-out)")

SFT split:  70 examples  (validation, all categories)
GRPO split: 1000 examples (test, physics)
Eval split: 200 examples  (test, physics, held-out)


## Step 6: Save to Disk

`cot_content` is saved only for the SFT split — GRPO and eval only need the question, options, and answer. This keeps the GRPO and eval files schema-clean and avoids confusion about which field drives training.

In [9]:
import os
import json

os.makedirs("physics_rlvr/data", exist_ok=True)

SFT_FIELDS  = ["question", "options", "answer", "cot_content"]
GRPO_FIELDS = ["question", "options", "answer"]
EVAL_FIELDS = ["question", "options", "answer"]

def save_jsonl(records, path, fields):
    with open(path, "w") as f:
        for rec in records:
            row = {k: rec[k] for k in fields}
            f.write(json.dumps(row) + "\n")
    print(f"Saved {len(records)} records -> {path}")

save_jsonl(sft_list,   "physics_rlvr/data/sft_train.jsonl",  SFT_FIELDS)
save_jsonl(grpo_split, "physics_rlvr/data/grpo_train.jsonl", GRPO_FIELDS)
save_jsonl(eval_split, "physics_rlvr/data/eval.jsonl",       EVAL_FIELDS)

Saved 70 records -> physics_rlvr/data/sft_train.jsonl
Saved 1000 records -> physics_rlvr/data/grpo_train.jsonl
Saved 200 records -> physics_rlvr/data/eval.jsonl


## Step 7: Sanity Check

In [10]:
print("=== SFT example ===")
ex = sft_list[0]
print(f"category:    {ex['category']}")
print(f"question:    {ex['question'][:80]}...")
print(f"options:     {ex['options'][:3]} ...")
print(f"answer:      {ex['answer']}")
print(f"cot_content: {ex['cot_content'][:150]}...")

print("\n=== GRPO example ===")
ex = grpo_split[0]
print(f"question: {ex['question'][:80]}...")
print(f"options:  {ex['options'][:3]} ...")
print(f"answer:   {ex['answer']}")

print("\n=== Eval example ===")
ex = eval_split[0]
print(f"question: {ex['question'][:80]}...")
print(f"options:  {ex['options'][:3]} ...")
print(f"answer:   {ex['answer']}")

print("\nData preparation complete.")

=== SFT example ===
category:    math
question:    The symmetric group $S_n$ has $
\factorial{n}$ elements, hence it is not true th...
options:     ['0', '30', '3'] ...
answer:      A
cot_content: A: Let's think step by step. A characteristic of a ring is R is $n$ if the statement $ka = 0$ for all $a\in 2Z$ implies that $k$ is a multiple of $n$....

=== GRPO example ===
question: (a) What is the acceleration of gravity on the moon's surface, g_m (as compared ...
options:  ['g_m = (1/6)g_e, 12 ft', 'g_m = (1/7)g_e, 16 ft', 'g_m = (1/4)g_e, 15 ft'] ...
answer:   A

=== Eval example ===
question: Two spheres of net charge +5e and -6e briefly come into contact. Afterward, whic...
options:  ['Both of the above', '+3e and -4e', '+5e and -6e'] ...
answer:   B

Data preparation complete.
